# 冻结主干网络

---

### 1️⃣ 主干网络（Backbone）是什么

* 在像 ResNet 这样的卷积神经网络（CNN）里，**前面的大部分层**都是用来提取特征的，比如边缘、颜色分布、纹理、形状等。
* 这些层的组合就叫做 **主干网络（Backbone）**，它相当于一个**通用的特征提取器**。
* 在 ResNet-18 里，主干部分是从 **第一个卷积层到最后一个卷积 block** 的全部卷积运算，不包括最后的全连接层（`fc`）。

📌 比喻：
主干网络 = 相机镜头
全连接层 = 镜头后的分类器（决定这张图是猫还是狗）

---

### 2️⃣ 冻结（Freeze）是什么意思

* **冻结参数** = 不让这些层的权重更新（梯度不计算），保持它们在预训练时学到的特征。
* 在 PyTorch 里实现很简单：

```python
for param in model.parameters():
param.requires_grad = False
```
* 冻结后，即使调用 `loss.backward()`，这些层的梯度也不会被计算，更不会被优化器更新。

📌 比喻：
冻结镜头（主干网络） → 不去磨镜头的玻璃，只换后面的“标签贴纸机”（全连接层）。

---

### 3️⃣ 为什么要冻结主干

1. **节省计算**：只训练最后一层，全网络参数量减少很多，训练更快。
2. **防止过拟合**：尤其是在小数据集上，保留大规模数据训练好的特征，减少因数据不足导致的过拟合。
3. **利用已有知识**：主干网络已在大数据集（如 ImageNet）上学到通用特征，不必重复学习。

---

### 4️⃣ 微调全连接层（Fine-tuning FC Layer）

* 因为你的任务类别数可能和预训练模型不一样（ImageNet 是 1000 类，你可能是 10 类），所以要**替换全连接层**：

```python
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
```
* 然后只训练这个新的全连接层，让它把“旧特征”映射到“新任务类别”。

---

### 5️⃣ 流程图示（逻辑顺序）

```
输入图像
↓
[冻结] 卷积层1 → 卷积层2 → ... → 卷积层N（主干网络）
↓
[训练] 新的全连接层（分类头）
↓
预测结果（新任务类别）
```

---

### 📌 ResNet-18 网络结构层次

```
输入：224 × 224 × 3
│
├── conv1: 7×7 卷积, 64通道, stride=2
├── bn1: 批归一化
├── ReLU
├── maxpool: 3×3 最大池化, stride=2
│
├── layer1:
│ ├── BasicBlock × 2 （64通道）
│
├── layer2:
│ ├── BasicBlock × 2 （128通道, 第一个 block stride=2）
│
├── layer3:
│ ├── BasicBlock × 2 （256通道, 第一个 block stride=2）
│
├── layer4:
│ ├── BasicBlock × 2 （512通道, 第一个 block stride=2）
│
├── avgpool: 全局平均池化
│
├── fc: 全连接层（512 → num_classes） ← **微调部分**
```

---

###  冻结主干 & 微调部分说明

* **主干网络（Backbone）**：

```
conv1, bn1, layer1, layer2, layer3, layer4, avgpool
```

这些层被冻结参数（`requires_grad=False`）。
* **微调部分（Head）**：

```
fc
```

替换成输出 `num_classes` 的新全连接层，只训练它的参数。

---

### 🖼️ 结构示意图（ASCII 版本）

```
[ 输入图像 224x224x3 ]
│
▼
[ conv1 7x7,64 ]
│
[ bn1 + ReLU ]
│
[ maxpool 3x3 ]
│
┌────────────── 主干网络（冻结） ────────────────┐
│ layer1: BasicBlock × 2 (64) │
│ layer2: BasicBlock × 2 (128) │
│ layer3: BasicBlock × 2 (256) │
│ layer4: BasicBlock × 2 (512) │
│ avgpool │
└──────────────────────────────────────────────┘
│
▼
[ fc: 512 → num_classes ] ← 仅训练这部分（微调）
```

---

### 冻结主干的目的

1. **预训练模型**

* 不是从你手里的任务开始训练，而是**别人**已经用**一个超大规模数据集**（如 ImageNet 1000 类，120 万张图片）训练好的模型。
* 这个模型的卷积层（主干网络）已经学到了通用的图像特征（边缘、颜色、纹理、形状等）。
* 这个阶段已经完成了，下载即用。

2. **冻结主干网络**

* 我们下载预训练模型后，可以**不更新**这些卷积层的参数（`requires_grad=False`），保持它原本的特征提取能力。

3. **微调全连接层（分类头）**

* 由于你的任务类别数、数据分布不同，需要替换全连接层，让它输出你任务的类别数。
* 只训练全连接层，让它学会用这些通用特征去解决你的具体任务。

---

### 总结：

> 预训练模型 = 别人已经训练好的“万能特征提取器” 
>
> 冻结主干网络 = 不去改动这个提取器的内部结构 
>
> 微调全连接层 = 只换“标签贴纸机”，让它贴上你任务需要的标签

---

In [1]:
# 导入必要的库
import torch  # PyTorch深度学习框架
import torch.nn as nn  # 神经网络模块
import torch.optim as optim  # 优化器
import torchvision  # 计算机视觉工具包
import torchvision.transforms as transforms  # 数据转换和增强
from torch.utils.data import DataLoader  # 数据加载器
from tqdm import tqdm  # 进度条
import matplotlib.pyplot as plt  # 绘图工具

# 检测并设置计算设备（GPU/CPU）
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
def get_data_loaders(batch_size=128):
    # 定义训练集的数据增强和预处理流程
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),  # 随机裁剪，增加模型鲁棒性
        transforms.RandomHorizontalFlip(),  # 随机水平翻转
        transforms.ColorJitter(brightness=0.2, contrast=0.2),  # 亮度和对比度增强
        transforms.RandomRotation(15),  # 随机旋转
        transforms.ToTensor(),  # 转换为张量
        transforms.Normalize((0.4914, 0.4822, 0.4465),  # CIFAR10数据集的标准化参数
                           (0.2023, 0.1994, 0.2010))
    ])
    
    # 定义测试集的数据预处理流程（不需要数据增强）
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), 
                           (0.2023, 0.1994, 0.2010))
    ])
    
    # 加载CIFAR10训练集
    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train)
    trainloader = DataLoader(
        trainset, batch_size=batch_size, shuffle=True, 
        num_workers=2, pin_memory=True)  # 启用pin_memory加速数据传输
    
    # 加载CIFAR10测试集
    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test)
    testloader = DataLoader(
        testset, batch_size=batch_size, shuffle=False, 
        num_workers=2, pin_memory=True)
    
    return trainloader, testloader

In [3]:
def get_model():
    # 加载预训练的ResNet18模型
    model = torchvision.models.resnet18(pretrained=True)
    
    # 选择性解冻层：只训练layer3、layer4和fc层
    for name, param in model.named_parameters():
        if "layer3" in name or "layer4" in name or "fc" in name:
            param.requires_grad = True  # 这些层将参与训练
        else:
            param.requires_grad = False  # 其他层保持冻结
    
    # 修改第一个卷积层以适应CIFAR10的32x32输入尺寸
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()  # 移除maxpool层，因为输入图像较小
    
    # 重新设计分类器结构
    model.fc = nn.Sequential(
        nn.Linear(512, 256),  # 降维
        nn.BatchNorm1d(256),  # 批归一化提升训练稳定性
        nn.ReLU(),  # 激活函数
        nn.Dropout(0.5),  # dropout防止过拟合
        nn.Linear(256, 10)  # 输出层，10个类别
    )
    
    return model.to(device)  # 将模型转移到指定设备

In [4]:
def train_epoch(model, trainloader, criterion, optimizer):
    model.train()  # 设置为训练模式
    running_loss = 0.0
    correct = 0
    total = 0
    
    # 使用tqdm显示训练进度
    pbar = tqdm(trainloader, desc='Training')
    for inputs, targets in pbar:
        # 将数据转移到指定设备
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()  # 清空梯度
        outputs = model(inputs)  # 前向传播
        loss = criterion(outputs, targets)  # 计算损失
        loss.backward()  # 反向传播
        
        # 梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()  # 更新参数
        
        # 统计训练指标
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # 更新进度条信息
        pbar.set_postfix({
            'loss': '%.3f' % (running_loss / (pbar.n + 1)),
            'acc': '%.3f%%' % (100. * correct / total)
        })
    
    return running_loss / len(trainloader), 100. * correct / total

In [5]:
@torch.no_grad()  # 禁用梯度计算
def evaluate(model, testloader, criterion):
    model.eval()  # 设置为评估模式
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in testloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(testloader), 100. * correct / total

In [7]:
def main():
    # 设置训练超参数
    BATCH_SIZE = 128
    EPOCHS = 100
    LR = 0.001
    
    # 加载数据
    trainloader, testloader = get_data_loaders(BATCH_SIZE)
    model = get_model()
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW([
        {'params': [p for n, p in model.named_parameters() if 'fc' not in n], 
         'lr': LR * 0.1},  # 主干网络使用较小学习率
        {'params': model.fc.parameters(), 'lr': LR}  # 分类器使用较大学习率
    ], weight_decay=1e-4)  # 添加L2正则化
    
    # 学习率调度器
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2)
    
    # 初始化记录器
    train_losses = []
    train_accs = []
    test_losses = []
    test_accs = []
    
    # 早停设置
    best_acc = 0
    patience = 10
    patience_counter = 0
    
    # 训练循环
    for epoch in range(EPOCHS):
        print(f'\nEpoch: {epoch+1}/{EPOCHS}')
        
        # 训练和评估
        train_loss, train_acc = train_epoch(
            model, trainloader, criterion, optimizer)
        test_loss, test_acc = evaluate(model, testloader, criterion)
        
        scheduler.step()  # 更新学习率
        
        # 记录指标
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        print(f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2f}%')
        print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc:.2f}%')
        
        # 模型保存和早停
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), 'best_model.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print('Early stopping triggered')
                break
    
    # 绘制训练曲线
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.legend()
    plt.title('Loss Curves')
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train Acc')
    plt.plot(test_accs, label='Test Acc')
    plt.legend()
    plt.title('Accuracy Curves')
    
    plt.tight_layout()
    plt.savefig('training_curves.png')
    plt.show()

if __name__ == '__main__':
    main()

Files already downloaded and verified
Files already downloaded and verified

Epoch: 1/100


Training:   0%|          | 0/391 [00:08<?, ?it/s]


KeyboardInterrupt: 